# RBD / REM-sleep-without-atonia — Colab pipelineStreams overnight polysomnography recordings and reduces each to a small derivedfile (rectified EMG envelope + hypnogram).**Why it is built this way.** A raw recording is ~430 MB; what the analysis needsis a few MB. So each recording is downloaded to Colab's *ephemeral* disk,reduced, saved to *Drive*, and the raw file deleted. Disk use stays flat, and adisconnected session loses at most one recording — everything finished is alreadyin Drive.Run the cells in order. It is safe to re-run any cell, and safe to lose thesession at any point: re-running resumes.

## 1. Mount DriveDerived data goes here so it survives session loss.

In [ ]:
from google.colab import drivedrive.mount('/content/drive')

## 2. Get the codeSet `REPO_URL` if you have pushed this project to GitHub — that is the tidiestoption. Otherwise upload the `rbd-rwa` folder to your Drive and the fallbackpath is used.

In [ ]:
import os, sys, subprocessfrom pathlib import PathREPO_URL  = ""                                    # e.g. "https://github.com/<you>/rbd-rwa.git"DRIVE_SRC = "/content/drive/MyDrive/rbd-rwa"      # fallback: code uploaded to Driveif REPO_URL:    if not Path("/content/rbd-rwa").exists():        subprocess.run(["git", "clone", REPO_URL, "/content/rbd-rwa"], check=True)    else:        subprocess.run(["git", "-C", "/content/rbd-rwa", "pull"], check=False)    CODE = "/content/rbd-rwa"else:    CODE = DRIVE_SRCassert Path(CODE, "src", "config.py").exists(), (    f"Could not find the code at {CODE}. Set REPO_URL, or upload the project "    f"folder to {DRIVE_SRC}")sys.path.insert(0, CODE)os.chdir(CODE)print("code:", CODE)

## 3. Install dependenciesColab already has numpy/scipy/pandas; MNE is the one addition.

In [ ]:
!pip install -q "mne>=1.6"import mne; print("mne", mne.__version__)

## 4. Check where data will live`durable` must be inside Drive. `scratch` is Colab-local and is wiped when thesession ends — that is intentional, it only ever holds the recording currentlybeing processed.

In [ ]:
from src import configconfig.describe()

## 5. ExtractDownloads, reduces, and deletes each raw recording in turn. The default cohort isthe 22 RBD patients plus 16 healthy controls from the CAP Sleep Database.If the session dies, just run this cell again — it continues from the manifest.

In [ ]:
!python run_extract.py

### Or a subset first, to check everything works end to end

In [ ]:
!python run_extract.py rbd1 n1

## 6. Inspect what was produced

In [ ]:
from src.extract import summarisefrom src import configfiles = sorted(config.DERIVED.glob("*.npz"))print(f"{len(files)} derived recordings")for f in files:    print(summarise(f))total = sum(f.stat().st_size for f in files)print(f"total derived size: {total/1e6:.0f} MB")

### Verify the clock alignmentThe annotation files carry wall-clock times while the signal starts whenever theEDF header says. A wrong offset would shift every sleep stage against the signaland silently corrupt the whole analysis, so check it explicitly: REM latencyshould be tens of minutes to a few hours after *scoring* begins, never zero.

In [ ]:
import numpy as npfrom src.extract import load_derivedfor f in sorted(config.DERIVED.glob("*.npz")):    env, hyp, meta = load_derived(f)    a = meta["alignment"]    rem = np.flatnonzero(hyp == 4)    scored = np.flatnonzero(hyp >= 0)    lat = (rem[0] - scored[0]) * 0.5 if len(rem) and len(scored) else float("nan")    flag = "" if lat > 15 else "  <-- CHECK"    print(f"{meta['record']:<8} offset {a['first_annotation_offset_s']/60:>7.1f} min   "          f"placed {a['n_placed']}/{a['n_annotated']}   "          f"REM latency {lat:>6.1f} min{flag}")